# Palintropos CUDA regression suite

In Colab, select a GPU runtime and choose **Runtime → Run all**. By default the notebook checks out the current head of PR #29's branch and records the exact SHA. Set `REF` in cell 1 to test a different branch or commit.

**Routine** runs `pytest -m "not slow"`, including short Held–Suarez regressions. Set `SUITE = "slow"` in cell 2 to run the long convergence checks separately on the same checkout, or use `"all"` for both. Standalone benchmark scripts are outside pytest. The separate architecture snapshot check runs on Python 3.12 in GitHub Actions.

The evidence ZIP includes the commit, environment, pytest logs, JUnit report when available, and skip reasons. Cell 3 downloads it even if pytest fails. For routine and all runs, the notebook requires the named CUDA coupling and frictional-heating tests to pass and flags skips in the radiative test module. Browser tests may skip when Chromium is unavailable. The evidence applies to the Colab Linux/CUDA environment printed by cell 1.


In [ ]:
# 1. Check out and prepare the selected revision.
import json
from pathlib import Path
import subprocess
import sys
import tempfile

REF = "feat/radiative-dynamical-foundation"
REPOSITORY = "https://github.com/AlexandreEros/Palintropos.git"

work = Path(tempfile.mkdtemp(prefix="palintropos-cuda-"))
repo = work / "repo"
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"], check=True)
subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPOSITORY, str(repo)], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin", REF], cwd=repo, check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"], cwd=repo, check=True)
sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(f"Testing {REF} at {sha}", flush=True)

# Reuse Colab's CuPy and let the runtime's pip install into the isolated environment.
def run_setup(command, cwd=None):
    result = subprocess.run(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    if result.returncode:
        print(result.stdout, end="", flush=True)
        result.check_returncode()

venv = work / "venv"
run_setup([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(venv)])
python = str(venv / "bin/python")
pip = [sys.executable, "-m", "pip", "--python", python]
requirements = [
    line for line in (repo / "requirements.txt").read_text().splitlines()
    if not line.lstrip().lower().startswith("cupy")
]
requirements += [
    line for line in (repo / "requirements-dev.txt").read_text().splitlines()
    if line.startswith(("grimp==", "networkx=="))
]
requirements += ["pytest", "ipykernel", "netCDF4==1.7.4"]
deps = work / "requirements-colab.txt"
deps.write_text("\n".join(requirements) + "\n")
run_setup([*pip, "install", "-q", "-r", str(deps)])
run_setup([*pip, "install", "-q", "--no-deps", "-e", "."], cwd=repo)

probe = r'''
import json, platform
import cupy as cp
import numpy as np
import pytest
import scipy
import tropoi

cp.cuda.runtime.getDeviceCount()
assert cp.arange(4).sum().item() == 6
cp.cuda.get_current_stream().synchronize()
print(json.dumps({
    "python": platform.python_version(), "platform": platform.platform(),
    "numpy": np.__version__, "scipy": scipy.__version__, "cupy": cp.__version__,
    "pytest": pytest.__version__,
    "cuda_driver": cp.cuda.runtime.driverGetVersion(),
    "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
    "tropoi_path": str(tropoi.__file__),
}))
'''
environment = json.loads(subprocess.check_output([python, "-c", probe], cwd=repo, text=True))
assert Path(environment["tropoi_path"]).is_relative_to(repo / "src"), environment["tropoi_path"]
environment["gpu"] = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print(json.dumps(environment, indent=2))


In [ ]:
# 2. Run pytest and prepare the evidence ZIP.
from collections import Counter
from datetime import datetime, timezone
import os
import shutil
import signal
from threading import Event, Timer
import xml.etree.ElementTree as ET

SUITE = "routine"
TIMEOUT_MINUTES = None  # Optional wall-clock limit for pytest.

selections = {"routine": ["-m", "not slow"], "slow": ["-m", "slow"], "all": []}
if SUITE not in selections:
    raise ValueError("SUITE must be routine, slow, or all")
if TIMEOUT_MINUTES is not None and TIMEOUT_MINUTES <= 0:
    raise ValueError("TIMEOUT_MINUTES must be positive or None")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
report = work / f"palintropos-cuda-{sha[:12]}-{SUITE}-{stamp}"
report.mkdir()
status = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"], cwd=repo, text=True
).strip()
if status:
    raise RuntimeError(f"Tracked files changed before testing:\n{status}")

junit = report / "pytest.xml"
command = [
    python, "-m", "pytest", "-q", "-ra", "--durations=20",
    f"--junitxml={junit}", *selections[SUITE],
]
record = dict(ref=REF, commit=sha, selection=SUITE, environment=environment,
              pytest_command=command, timeout_minutes=TIMEOUT_MINUTES)
timed_out = Event()
print("Running:", " ".join(command), flush=True)
with (report / "pytest.log").open("w", encoding="utf-8") as output:
    with subprocess.Popen(
        command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", start_new_session=True,
        env=dict(os.environ, PYTHONUNBUFFERED="1"),
    ) as process:
        def stop_at_deadline():
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                return
            timed_out.set()

        deadline = None
        if TIMEOUT_MINUTES is not None:
            deadline = Timer(TIMEOUT_MINUTES * 60, stop_at_deadline)
            deadline.daemon = True
            deadline.start()
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                output.write(line)
                output.flush()
            record["pytest_exit_code"] = process.wait()
        finally:
            if deadline is not None:
                deadline.cancel()
                deadline.join()

counts = Counter()
skipped = []
radiative = []
problems = []
record["timed_out"] = timed_out.is_set()
if record["timed_out"]:
    problems.append("Pytest exceeded the configured wall-clock limit.")
elif record["pytest_exit_code"] != 0:
    problems.append(f"Pytest exited with code {record['pytest_exit_code']}.")

try:
    root = ET.parse(junit).getroot()
except (OSError, ET.ParseError) as error:
    problems.append(f"No readable JUnit report: {error}")
else:
    for case in root.iter("testcase"):
        module = case.get("classname", "")
        name = case.get("name", "")
        skip = case.find("skipped")
        outcome = next(
            (kind for kind in ("error", "failure", "skipped") if case.find(kind) is not None),
            "passed",
        )
        counts[outcome] += 1
        if skip is not None:
            skipped.append(dict(module=module, name=name, reason=skip.get("message") or skip.text))
        if module.rsplit(".", 1)[-1] == "test_radiative_dynamical":
            radiative.append(dict(name=name, outcome=outcome))

if SUITE != "slow":
    required = {
        "test_coupled_energy_budget_closes_and_a_resumed_run_is_identical",
        "test_frictional_heating_returns_what_hyperdiffusion_removes_adiabatic",
        "test_with_frictional_heating_the_coupled_budget_closes_with_hyperdiffusion_on",
    }
    passed = {case["name"].split("[", 1)[0] for case in radiative if case["outcome"] == "passed"}
    missing = sorted(required - passed)
    if missing:
        problems.append("Required CUDA tests did not pass: " + ", ".join(missing))
    if any(case["outcome"] != "passed" for case in radiative):
        problems.append("At least one radiative regression failed or skipped. See radiative_tests.")

record.update(pytest_counts=dict(counts), skipped=skipped, radiative_tests=radiative,
              passed=not problems, problems=problems)
(report / "result.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
archive = shutil.make_archive(str(report), "zip", root_dir=report)
print(json.dumps(record, indent=2))
print(f"Evidence ZIP: {archive}")


In [ ]:
# 3. Download the evidence ZIP and report the final status.
from google.colab import files

files.download(archive)
if not record["passed"]:
    raise RuntimeError("CUDA validation did not pass. See result.json and pytest.log in the ZIP.")
